In [3]:
# ===============================
#   ВАШИ ДАННЫЕ — ВВЕДИТЕ ЗДЕСЬ
# ===============================

# Пример: просто замените цифры на свои Viability
ctrl_day1 = [100, 83, 67, 64, 63, 71, 86, 86]
ctrl_day2 = [88, 33, 60, 50, 25, 79, 83, 25, 70]
ctrl_day3 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

drug_day1 = [92, 60, 83, 100, 91, 100, 100, 100]
drug_day2 = [100, 68, 88, 100, 100, 100]
drug_day3 = [60, 77, 50, 91, 60, 75, 67, 69, 90, 25]
drug_day4 = [0, 0, 0, 0, 0, 0]

# ===============================
#    АВТОМАТИЧЕСКИЙ АНАЛИЗ
# ===============================

import numpy as np
import pandas as pd
import scipy.stats as stats
import scikit_posthocs as sp
from statsmodels.stats.multitest import multipletests

# ---------------------------------------
# Helper: звездочки по p-value
# ---------------------------------------
def stars(p):
    if p < 0.0001: return "****"
    elif p < 0.001: return "***"
    elif p < 0.01: return "**"
    elif p < 0.05: return "*"
    else: return "ns"

# ---------------------------------------
# Сбор данных в единый DataFrame
# ---------------------------------------
dat = []

for day, values in [(1, ctrl_day1),(2, ctrl_day2),(3, ctrl_day3)]:
    for v in values:
        dat.append(["ctrl", day, v])

for day, values in [(1, drug_day1),(2, drug_day2),(3, drug_day3),(4, drug_day4)]:
    for v in values:
        dat.append(["drug", day, v])

dat = pd.DataFrame(dat, columns=["group", "day", "value"])

# ---------------------------------------
# 1. Kruskal–Wallis + Dunn post-hoc
# ---------------------------------------

def analyze_group(name):
    sub = dat[dat["group"] == name]
    days = sorted(sub["day"].unique())

    # Kruskal–Wallis
    groups = [sub[sub["day"]==d]["value"] for d in days]
    H, p_kw = stats.kruskal(*groups)

    print(f"\n### {name.upper()} — Kruskal–Wallis")
    print(f"H = {H:.3f}, p = {p_kw:.4f}, {stars(p_kw)}")

    # Dunn post-hoc
    print(f"\n{name.upper()} — Dunn post-hoc (Holm):")
    dunn = sp.posthoc_dunn(sub, val_col="value", group_col="day", p_adjust="holm")
    
    # Добавляем звездочки
    dunn_stars = dunn.copy()
    for r in dunn.index:
        for c in dunn.columns:
            dunn_stars.loc[r,c] = stars(dunn.loc[r,c])

    print("\nP-values:")
    print(dunn)
    print("\nStars:")
    print(dunn_stars)

analyze_group("ctrl")
analyze_group("drug")

# ---------------------------------------
# 2. Mann–Whitney между группами по дням
# ---------------------------------------

print("\n### MANN–WHITNEY BETWEEN GROUPS ###")

mw_results = []
days_all = sorted(dat["day"].unique())

for d in days_all:
    c = dat[(dat.group=="ctrl") & (dat.day==d)].value
    dr = dat[(dat.group=="drug") & (dat.day==d)].value
    if len(c)==0 or len(dr)==0:
        continue
    U, p = stats.mannwhitneyu(c, dr, alternative="two-sided")
    mw_results.append([d, p])

# коррекция Holm
raw_p = [x[1] for x in mw_results]
_, p_corr, _, _ = multipletests(raw_p, method="holm")

# таблица
mw_table = pd.DataFrame({
    "day": [x[0] for x in mw_results],
    "p_raw": raw_p,
    "p_holm": p_corr,
    "stars": [stars(p) for p in p_corr]
})

print("\nMann–Whitney results:")
print(mw_table)


### CTRL — Kruskal–Wallis
H = 20.409, p = 0.0000, ****

CTRL — Dunn post-hoc (Holm):

P-values:
          1         2         3
1  1.000000  0.271371  0.000057
2  0.271371  1.000000  0.002283
3  0.000057  0.002283  1.000000

Stars:
      1   2     3
1    ns  ns  ****
2    ns  ns    **
3  ****  **    ns

### DRUG — Kruskal–Wallis
H = 20.878, p = 0.0001, ***

DRUG — Dunn post-hoc (Holm):

P-values:
          1         2         3         4
1  1.000000  0.848466  0.123075  0.000504
2  0.848466  1.000000  0.123075  0.000573
3  0.123075  0.123075  1.000000  0.123075
4  0.000504  0.000573  0.123075  1.000000

Stars:
     1    2   3    4
1   ns   ns  ns  ***
2   ns   ns  ns  ***
3   ns   ns  ns   ns
4  ***  ***  ns   ns

### MANN–WHITNEY BETWEEN GROUPS ###

Mann–Whitney results:
   day     p_raw    p_holm stars
0    1  0.087600  0.087600    ns
1    2  0.008768  0.017537     *
2    3  0.000063  0.000190   ***


In [2]:
pip install scikit-posthocs

Note: you may need to restart the kernel to use updated packages.


In [7]:
# ===============================
#   ВАШИ ДАННЫЕ — ВВЕДИТЕ ЗДЕСЬ
# ===============================

# Пример: просто замените цифры на свои Cell Size
ctrl_day1 = [91.092, 93.482, 54.058, 88.106, 41.813, 115.284, 127.828, 82.132, 
             45.695, 41.216, 54.357, 43.008, 116.18, 61.823, 45.695, 35.84, 
             120.361, 46.591, 58.538, 132.905, 99.754, 69.887, 66.901, 64.81, 
             20.309, 60.927, 37.333, 36.138, 32.256, 38.826, 56.149, 66.602, 
             68.394, 65.407, 27.477, 85.119, 37.93, 46.293, 62.421, 76.458, 
             69.887, 34.944, 26.581, 22.997, 38.229, 38.528, 61.226, 
             64.511, 35.541, 100.052, 61.226, 72.277, 54.655, 31.658, 97.663, 79.146, 
             84.522, 65.407, 41.514, 67.199, 
             29.269, 26.282, 40.021, 89.599, 124.244, 22.997, 138.281, 163.369, 
             48.682, 69.887, 65.706, 123.348, 
             38.229, 39.125, 57.343]
ctrl_day2 = [49.279, 121.556, 147.54, 74.069, 131.113, 311.506, 152.617, 56.447, 217.128, 
             82.73, 281.938, 285.821, 240.125, 272.978, 83.924, 203.987, 142.761, 232.957, 
             213.544, 153.214, 30.464, 103.636, 287.613, 185.47, 
             103.935, 92.287, 57.642, 153.214, 245.501, 177, 243.411, 
             272.082, 177.1075, 98.26, 175.315, 91.391, 64.81, 
             356.305, 239.827, 108.116, 145.15, 106.921, 123.049]

#ctrl_day3 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

drug_day1 = [152.02, 109.311, 75.86, 196.819, 70.783, 68.095,
             24.789, 216.232, 68.693, 56.447, 25.386, 85.418, 80.639, 31.36, 50.773, 87.21, 
             89.002, 65.407, 111.401, 80.639, 35.242, 144.254, 124.841, 186.067, 
             89.002, 24.789, 172.03, 104.532, 141.566, 102.442, 28.373, 156.798, 
             40.917, 26.282, 38.229, 85.418, 73.173, 30.762, 60.629, 149.332, 
             180.094, 46.293, 84.223, 58.538, 139.177, 48.085, 24.49, 135.593, 
             111.103, 99.754, 54.655, 70.186, 79.146, 66.005, 74.069, 
             97.364, 105.428, 52.565, 103.338, 61.226, 84.82, 54.954, 
             79.743, 54.655, 105.727, 60.031, 69.589, 64.511, 115.881, 38.528, 68.394, 66.602, 
             96.17, 91.09, 96.468, 45.695, 91.09, 59.135, 72.575, 75.562, 52.266]
drug_day2 = [27.178, 32.853, 15.829, 44.501, 94.079, 144.254, 73.77, 40.021, 80.938, 22.101, 71.679, 115.284, 
             84.37, 74.666, 83.626, 66.005, 95.274, 74.964, 84.37, 108.415, 73.173, 93.78, 19.413, 89.002, 
             68.991, 83.028, 111.103, 173.225, 131.412, 96.17, 97.962, 108.713, 63.018, 134.84, 62.122, 143.956, 
             23.893, 166.057, 44.202, 134.84, 180.691, 41.216, 115.881, 77.055, 129.321, 130.815, 134.1, 
             180.393, 72.575, 42.112, 51.967, 84.223, 79.444, 149.033, 250.579, 24.192, 207.272, 83.924, 57.045, 
             61.226, 96.468, 125.737, 82.73, 92.884, 67.498, 82.431, 51.669, 165.459]
drug_day3 = [182.185, 44.202, 232.061, 100.052, 261.629, 343.463, 134.1, 157.097, 116.777, 56.149, 
             86.612, 108.116, 40.917, 70.485, 61.226, 299.559, 33.749, 72.575, 
             68.095, 291.794, 260.733, 76.159, 150.526, 71.978, 151.721, 198.312, 
             152.61, 345.852, 101.546, 160.681, 290.002, 147.838, 128.127, 53.162, 
             152.61, 216.531, 191.742, 41.514, 52.565, 170.835, 54.655, 
             108.116, 38.229, 39.125, 187.859, 25.088, 138.878, 108.713, 
             184.275, 141.268, 68.991, 122.452, 33.152, 
             211.752, 192.937, 76.159, 46.89, 203.39, 
             195.923, 34.048, 84.223, 188.158, 193.534, 
             152.617, 176.51]
#drug_day4 = [0, 0, 0, 0, 0, 0]

# ===============================
#    АВТОМАТИЧЕСКИЙ АНАЛИЗ
# ===============================

import numpy as np
import pandas as pd
import scipy.stats as stats
import scikit_posthocs as sp
from statsmodels.stats.multitest import multipletests

ctrl_mean = np.mean(ctrl_day1)
drug_mean = np.mean(drug_day1)

ctrl_day1 = ctrl_day1/np.mean(ctrl_mean)
ctrl_day2 = ctrl_day2/np.mean(ctrl_mean)
drug_day1 = drug_day1/np.mean(drug_mean)
drug_day2 = drug_day2/np.mean(drug_mean)
drug_day3 = drug_day3/np.mean(drug_mean)
# ---------------------------------------
# Helper: звездочки по p-value
# ---------------------------------------
def stars(p):
    if p < 0.0001: return "****"
    elif p < 0.001: return "***"
    elif p < 0.01: return "**"
    elif p < 0.05: return "*"
    else: return "ns"

# ---------------------------------------
# Сбор данных в единый DataFrame
# ---------------------------------------
dat = []

for day, values in [(1, ctrl_day1),(2, ctrl_day2)]:
    for v in values:
        dat.append(["ctrl", day, v])

for day, values in [(1, drug_day1),(2, drug_day2),(3, drug_day3)]:
    for v in values:
        dat.append(["drug", day, v])

dat = pd.DataFrame(dat, columns=["group", "day", "value"])

# ---------------------------------------
# 1. Kruskal–Wallis + Dunn post-hoc
# ---------------------------------------

def analyze_group(name):
    sub = dat[dat["group"] == name]
    days = sorted(sub["day"].unique())

    # Kruskal–Wallis
    groups = [sub[sub["day"]==d]["value"] for d in days]
    H, p_kw = stats.kruskal(*groups)

    print(f"\n### {name.upper()} — Kruskal–Wallis")
    print(f"H = {H:.3f}, p = {p_kw:.4f}, {stars(p_kw)}")

    # Dunn post-hoc
    print(f"\n{name.upper()} — Dunn post-hoc (Holm):")
    dunn = sp.posthoc_dunn(sub, val_col="value", group_col="day", p_adjust="holm")
    
    # Добавляем звездочки
    dunn_stars = dunn.copy()
    for r in dunn.index:
        for c in dunn.columns:
            dunn_stars.loc[r,c] = stars(dunn.loc[r,c])

    print("\nP-values:")
    print(dunn)
    print("\nStars:")
    print(dunn_stars)

analyze_group("ctrl")
analyze_group("drug")

# ---------------------------------------
# 2. Mann–Whitney между группами по дням
# ---------------------------------------

print("\n### MANN–WHITNEY BETWEEN GROUPS ###")

mw_results = []
days_all = sorted(dat["day"].unique())

for d in days_all:
    c = dat[(dat.group=="ctrl") & (dat.day==d)].value
    dr = dat[(dat.group=="drug") & (dat.day==d)].value
    if len(c)==0 or len(dr)==0:
        continue
    U, p = stats.mannwhitneyu(c, dr, alternative="two-sided")
    mw_results.append([d, p])

# коррекция Holm
raw_p = [x[1] for x in mw_results]
_, p_corr, _, _ = multipletests(raw_p, method="holm")

# таблица
mw_table = pd.DataFrame({
    "day": [x[0] for x in mw_results],
    "p_raw": raw_p,
    "p_holm": p_corr,
    "stars": [stars(p) for p in p_corr]
})

print("\nMann–Whitney results:")
print(mw_table)


### CTRL — Kruskal–Wallis
H = 47.693, p = 0.0000, ****

CTRL — Dunn post-hoc (Holm):

P-values:
              1             2
1  1.000000e+00  4.984324e-12
2  4.984324e-12  1.000000e+00

Stars:
      1     2
1    ns  ****
2  ****    ns

### DRUG — Kruskal–Wallis
H = 17.302, p = 0.0002, ***

DRUG — Dunn post-hoc (Holm):

P-values:
          1         2         3
1  1.000000  0.310968  0.000149
2  0.310968  1.000000  0.006675
3  0.000149  0.006675  1.000000

Stars:
     1   2    3
1   ns  ns  ***
2   ns  ns   **
3  ***  **   ns

### MANN–WHITNEY BETWEEN GROUPS ###

Mann–Whitney results:
   day         p_raw        p_holm stars
0    1  9.745328e-01  9.745328e-01    ns
1    2  3.124792e-10  6.249585e-10  ****


In [8]:
# ===============================
#   ВАШИ ДАННЫЕ — ВВЕДИТЕ ЗДЕСЬ
# ===============================

# Пример: замените на свои значения Cell Density Anova
ctrl_day1 = [14.81481481, 22.22222222, 11.11111111, 13.58024691, 9.87654321, 20.98765432, 17.28395062, 8.641975309]
ctrl_day2 = [9.87654321, 11.11, 6.172839506, 9.87654321, 9.87654321, 17.28395062, 14.81481481, 9.87654321, 12.34567901]
ctrl_day3 = [23.45679012, 9.87654321, 11.11111111, 23.45679012, 11.11111111, 12.34567901, 14.81481481, 13.58024691, 8.641975309, 
             11.11111111]

drug_day1 = [16.04938272, 12.34567901, 14.81481481, 9.87654321, 13.58024691, 17.28395062, 9.87654321, 16.04938272]
drug_day2 = [14.81481481, 27.16049383, 20.98765432, 12.34567901, 11.11111111, 8.641975309]
drug_day3 = [12.34567901, 16.04938272, 2.469135802, 13.58024691, 12.34567901, 14.81481481, 7.407407407, 16.04938272, 12.34567901, 
             4.938271605]
drug_day4 = [14.81481481, 16.04938272, 11.11111111, 11.11111111, 8.641975309, 13.58024691]

# ===============================
#    АВТОМАТИЧЕСКИЙ АНАЛИЗ
# ===============================

import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests

# ---------------------------------------
# Helper: звездочки по p-value
# ---------------------------------------
def stars(p):
    if p < 0.0001: return "****"
    elif p < 0.001: return "***"
    elif p < 0.01: return "**"
    elif p < 0.05: return "*"
    else: return "ns"

# ---------------------------------------
# Сбор данных в единый DataFrame
# ---------------------------------------
dat = []

for day, values in [(1, ctrl_day1),(2, ctrl_day2),(3, ctrl_day3)]:
    for v in values:
        dat.append(["ctrl", day, v])

for day, values in [(1, drug_day1),(2, drug_day2),(3, drug_day3),(4, drug_day4)]:
    for v in values:
        dat.append(["drug", day, v])

dat = pd.DataFrame(dat, columns=["group", "day", "value"])

# ---------------------------------------
# 1. ANOVA + Tukey post-hoc
# ---------------------------------------
def analyze_group(name):
    sub = dat[dat["group"] == name]
    days = sorted(sub["day"].unique())

    # ANOVA
    groups = [sub[sub["day"]==d]["value"] for d in days]
    F, p_anova = stats.f_oneway(*groups)
    print(f"\n### {name.upper()} — ANOVA")
    print(f"F = {F:.3f}, p = {p_anova:.4f}, {stars(p_anova)}")

    # Tukey HSD
    print(f"\n{name.upper()} — Tukey HSD post-hoc:")
    tukey = pairwise_tukeyhsd(sub["value"], sub["day"])
    print(tukey.summary())

    # Добавляем звездочки
    tukey_df = pd.DataFrame(data=tukey._results_table.data[1:], columns=tukey._results_table.data[0])
    tukey_df["stars"] = tukey_df["p-adj"].apply(stars)
    print("\nTukey post-hoc с звездочками:")
    print(tukey_df)

analyze_group("ctrl")
analyze_group("drug")

# ---------------------------------------
# 2. t-test между группами по дням
# ---------------------------------------
print("\n### T-TEST BETWEEN GROUPS ###")

mw_results = []
days_all = sorted(dat["day"].unique())

for d in days_all:
    c = dat[(dat.group=="ctrl") & (dat.day==d)]["value"]
    dr = dat[(dat.group=="drug") & (dat.day==d)]["value"]
    if len(c)==0 or len(dr)==0:
        continue
    t_stat, p = stats.ttest_ind(c, dr)
    mw_results.append([d, p])

# коррекция Holm для нескольких дней
raw_p = [x[1] for x in mw_results]
_, p_corr, _, _ = multipletests(raw_p, method='holm')

# таблица
ttest_table = pd.DataFrame({
    "day": [x[0] for x in mw_results],
    "p_raw": raw_p,
    "p_holm": p_corr,
    "stars": [stars(p) for p in p_corr]
})

print("\nt-test results:")
print(ttest_table)


### CTRL — ANOVA
F = 1.413, p = 0.2629, ns

CTRL — Tukey HSD post-hoc:
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
     1      2  -3.5667 0.2708 -9.1792 2.0459  False
     1      3  -0.8642    0.9 -6.3431 4.6147  False
     2      3   2.7025 0.4261 -2.6047 8.0096  False
---------------------------------------------------

Tukey post-hoc с звездочками:
   group1  group2  meandiff   p-adj   lower   upper  reject stars
0       1       2   -3.5667  0.2708 -9.1792  2.0459   False    ns
1       1       3   -0.8642  0.9000 -6.3431  4.6147   False    ns
2       2       3    2.7025  0.4261 -2.6047  8.0096   False    ns

### DRUG — ANOVA
F = 1.375, p = 0.2724, ns

DRUG — Tukey HSD post-hoc:
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
     1      2   2.1091 0.8061  -4.6159

In [9]:
# ===============================
#   ВАШИ ДАННЫЕ — ВВЕДИТЕ ЗДЕСЬ
# ===============================

# Пример: просто замените цифры на Cell Density not Anova

ctrl_day1 = [14.81481481, 22.22222222, 11.11111111, 13.58024691, 9.87654321, 20.98765432, 17.28395062, 8.641975309]
ctrl_day2 = [9.87654321, 11.11, 6.172839506, 9.87654321, 9.87654321, 17.28395062, 14.81481481, 9.87654321, 12.34567901]
ctrl_day3 = [23.45679012, 9.87654321, 11.11111111, 23.45679012, 11.11111111, 12.34567901, 14.81481481, 13.58024691, 8.641975309, 
             11.11111111]

drug_day1 = [16.04938272, 12.34567901, 14.81481481, 9.87654321, 13.58024691, 17.28395062, 9.87654321, 16.04938272]
drug_day2 = [14.81481481, 27.16049383, 20.98765432, 12.34567901, 11.11111111, 8.641975309]
drug_day3 = [12.34567901, 16.04938272, 2.469135802, 13.58024691, 12.34567901, 14.81481481, 7.407407407, 16.04938272, 12.34567901, 
             4.938271605]
drug_day4 = [14.81481481, 16.04938272, 11.11111111, 11.11111111, 8.641975309, 13.58024691]

# ===============================
#    АВТОМАТИЧЕСКИЙ АНАЛИЗ
# ===============================

import numpy as np
import pandas as pd
import scipy.stats as stats
import scikit_posthocs as sp
from statsmodels.stats.multitest import multipletests

# ---------------------------------------
# Helper: звездочки по p-value
# ---------------------------------------
def stars(p):
    if p < 0.0001: return "****"
    elif p < 0.001: return "***"
    elif p < 0.01: return "**"
    elif p < 0.05: return "*"
    else: return "ns"

# ---------------------------------------
# Сбор данных в единый DataFrame
# ---------------------------------------
dat = []

for day, values in [(1, ctrl_day1),(2, ctrl_day2),(3, ctrl_day3)]:
    for v in values:
        dat.append(["ctrl", day, v])

for day, values in [(1, drug_day1),(2, drug_day2),(3, drug_day3),(4, drug_day4)]:
    for v in values:
        dat.append(["drug", day, v])

dat = pd.DataFrame(dat, columns=["group", "day", "value"])

# ---------------------------------------
# 1. Kruskal–Wallis + Dunn post-hoc
# ---------------------------------------

def analyze_group(name):
    sub = dat[dat["group"] == name]
    days = sorted(sub["day"].unique())

    # Kruskal–Wallis
    groups = [sub[sub["day"]==d]["value"] for d in days]
    H, p_kw = stats.kruskal(*groups)

    print(f"\n### {name.upper()} — Kruskal–Wallis")
    print(f"H = {H:.3f}, p = {p_kw:.4f}, {stars(p_kw)}")

    # Dunn post-hoc
    print(f"\n{name.upper()} — Dunn post-hoc (Holm):")
    dunn = sp.posthoc_dunn(sub, val_col="value", group_col="day", p_adjust="holm")
    
    # Добавляем звездочки
    dunn_stars = dunn.copy()
    for r in dunn.index:
        for c in dunn.columns:
            dunn_stars.loc[r,c] = stars(dunn.loc[r,c])

    print("\nP-values:")
    print(dunn)
    print("\nStars:")
    print(dunn_stars)

analyze_group("ctrl")
analyze_group("drug")

# ---------------------------------------
# 2. Mann–Whitney между группами по дням
# ---------------------------------------

print("\n### MANN–WHITNEY BETWEEN GROUPS ###")

mw_results = []
days_all = sorted(dat["day"].unique())

for d in days_all:
    c = dat[(dat.group=="ctrl") & (dat.day==d)].value
    dr = dat[(dat.group=="drug") & (dat.day==d)].value
    if len(c)==0 or len(dr)==0:
        continue
    U, p = stats.mannwhitneyu(c, dr, alternative="two-sided")
    mw_results.append([d, p])

# коррекция Holm
raw_p = [x[1] for x in mw_results]
_, p_corr, _, _ = multipletests(raw_p, method="holm")

# таблица
mw_table = pd.DataFrame({
    "day": [x[0] for x in mw_results],
    "p_raw": raw_p,
    "p_holm": p_corr,
    "stars": [stars(p) for p in p_corr]
})

print("\nMann–Whitney results:")
print(mw_table)


### CTRL — Kruskal–Wallis
H = 2.666, p = 0.2637, ns

CTRL — Dunn post-hoc (Holm):

P-values:
          1         2         3
1  1.000000  0.396875  0.775980
2  0.396875  1.000000  0.396875
3  0.775980  0.396875  1.000000

Stars:
    1   2   3
1  ns  ns  ns
2  ns  ns  ns
3  ns  ns  ns

### DRUG — Kruskal–Wallis
H = 1.671, p = 0.6435, ns

DRUG — Dunn post-hoc (Holm):

P-values:
     1    2    3    4
1  1.0  1.0  1.0  1.0
2  1.0  1.0  1.0  1.0
3  1.0  1.0  1.0  1.0
4  1.0  1.0  1.0  1.0

Stars:
    1   2   3   4
1  ns  ns  ns  ns
2  ns  ns  ns  ns
3  ns  ns  ns  ns
4  ns  ns  ns  ns

### MANN–WHITNEY BETWEEN GROUPS ###

Mann–Whitney results:
   day     p_raw    p_holm stars
0    1  0.832662  1.000000    ns
1    2  0.170672  0.512017    ns
2    3  0.819390  1.000000    ns


In [ ]:
import numpy as np


In [1]:
pip list


Package             Version
------------------- -----------
asttokens           2.0.5
backcall            0.2.0
colorama            0.4.6
comm                0.2.1
contourpy           1.1.1
cycler              0.12.1
debugpy             1.6.7
decorator           5.1.1
et_xmlfile          2.0.0
exceptiongroup      1.3.0
executing           0.8.3
fonttools           4.57.0
importlib-metadata  7.0.1
importlib_resources 6.4.5
iniconfig           2.1.0
ipyfilechooser      0.6.0
ipykernel           6.29.5
ipython             8.12.2
ipywidgets          8.1.7
jedi                0.19.1
jupyter_client      8.6.0
jupyter_core        5.7.2
jupyterlab_widgets  3.0.15
kiwisolver          1.4.7
matplotlib          3.7.5
matplotlib-inline   0.1.6
mkl-service         2.3.0
nest-asyncio        1.6.0
numpy               1.23.5
openpyxl            3.1.5
packaging           24.1
pandas              1.2.5
parso               0.8.3
patsy               0.5.6
pickleshare         0.7.5
pillow              10.4